This summary focuses exclusively on the provided transcript for **Lecture 3: Policy Evaluation**, which marks the transition from planning with known models to **model-free reinforcement learning**.

### **Summary of Lecture 3**
The core objective of this lecture is **model-free policy evaluation**: determining the value function ($V^\pi$) for a fixed policy when the agent is not given a dynamics model ($P$) or a reward model ($R$). The agent must learn how "good" a policy is through direct experience in the environment. The lecture explores two primary families of algorithms for this task: **Monte Carlo (MC) methods** and **Temporal Difference (TD) learning**, as well as how they behave when applied to a fixed batch of data.

### **Key Terms**
*   **Tabular MDP:** A setting where the state space is small enough that the value for each state can be stored as an entry in a table, rather than needing function approximation like a neural network.
*   **Monte Carlo (MC) Evaluation:** A method that estimates values by averaging the **Returns (G)** (the discounted sum of rewards) observed across multiple full episodes.
*   **Temporal Difference (TD) Learning:** A method that updates value estimates after every single step (state, action, reward, next state) rather than waiting for an episode to end.
*   **Bootstrapping:** The process of updating an estimate based on another existing estimate. TD learning "bootstraps" by using the current estimated value of the *next* state to update the value of the *current* state.
*   **TD Target:** The value used as the goal for an update in TD learning, defined as the immediate reward plus the discounted value of the next state ($R + \gamma V(S')$).
*   **Certainty Equivalence:** An approach where an agent uses its experience to build a maximum likelihood model of the environment (estimating $P$ and $R$ from counts) and then solves it using dynamic programming.

### **Key Concepts**
*   **The Markov Assumption:** A critical distinction between the two methods is that **Monte Carlo does not require the system to be Markovian**; it simply averages returns. Conversely, **TD learning exploits the Markov structure** by linking the value of a state to the value of its successor.
*   **First-Visit vs. Every-Visit MC:** In first-visit MC, a state's value is updated only the first time it is visited in an episode, resulting in an **unbiased** estimator. Every-visit MC updates every time a state is visited, which is biased but often has a better mean squared error because it uses more data.
*   **Bias vs. Variance Trade-off:** 
    *   **Monte Carlo** is generally unbiased but has **high variance** because the returns can differ significantly between episodes. 
    *   **TD learning** has **lower variance** because it updates step-by-step, though it is biased by the initial values of the value function.
*   **Batch Policy Evaluation:** When training repeatedly on a finite set of data:
    *   **MC converges** to the solution that minimizes the **mean squared error** relative to the observed returns.
    *   **TD(0) converges** to the **certainty equivalence solution**, which is the same result you would get if you built a model from the data and then solved it.
*   **Data Efficiency:** Certainty equivalence (and by extension TD in batch settings) is highly **data efficient** because it propagates information across all known transitions, whereas MC only updates states actually visited in a specific trajectory.

This coding exercise is designed to reinforce the core concepts of **model-free policy evaluation** discussed in the lecture, specifically the implementation of **Monte Carlo (MC)** and **Temporal Difference (TD)** learning in a tabular setting.

### **Exercise: Comparing MC and TD Evaluation**

**Objective:** Implement **First-Visit Monte Carlo** and **TD(0)** to evaluate a fixed policy in a simple 1D "Mars Rover" environment. In this environment, an agent moves along a line of 5 states. If it reaches the far right, it gets a reward of +1; otherwise, rewards are 0.

#### **1. Setup the Environment**

In [ ]:
import numpy as np

# Environment Setup
num_states = 5
terminal_states = [5]
# Fixed Policy: Always move right (action 1)
# Transitions: State s -> s+1 with probability 1.0 (deterministic)
gamma = 1.0  # Discount factor
alpha = 0.1  # Learning rate

#### **2. Task: Implement First-Visit Monte Carlo**
**Concept:** MC evaluation averages the **returns (G)** observed across full episodes. In the first-visit version, you only update a state's value the first time it appears in an episode to keep the estimator **unbiased**.


In [ ]:
def first_visit_mc(num_episodes):
    V = np.zeros(num_states)
    returns_sum = np.zeros(num_states)
    returns_count = np.zeros(num_states)

    for _ in range(num_episodes):
        # Generate one episode: (state, reward) tuples
        episode = [(1, 0), (2, 0), (3, 1)] # Example trajectory starting at S1
        
        # Track which states we've seen in this episode
        visited_in_episode = set()
        
        # Calculate returns from the end of the episode backward
        G = 0
        for s, r in reversed(episode):
            G = r + gamma * G
            if s not in visited_in_episode:
                # TODO: Update the running average for state 's'
                # Hint: Incremental update formula: V(s) = V(s) + alpha * (G - V(s))
                visited_in_episode.add(s)
                V[s] += alpha * (G - V[s])
    return V

#### **3. Task: Implement TD(0) Learning**
**Concept:** TD(0) updates the value of a state **immediately** after each step by **bootstrapping**. It uses the reward plus the estimated value of the *next* state as its target.

In [ ]:
def td_zero(num_episodes):
    V = np.zeros(num_states)
    
    for _ in range(num_episodes):
        state = 1 # Start state
        while state not in terminal_states:
            # Take action according to policy (Move Right)
            next_state = state + 1
            reward = 1 if next_state == 4 else 0
            
            # TODO: Implement the TD(0) Update
            # Target = Reward + gamma * V(next_state)
            # V(state) = V(state) + alpha * (Target - V(state))
            target = reward + gamma * V[next_state]
            V[state] += alpha * (target - V[state])
            
            state = next_state
    return V

---

### **Conceptual Review Questions**
1.  **Episodic vs. Continuous:** Why can't you use Monte Carlo evaluation for a robot that is designed to act forever without ever reaching a "terminal" state?
2.  **Bootstrapping:** In the code above, which algorithm uses "bootstrapping," and what does that term specifically mean regarding how estimates are updated?
3.  **Efficiency:** If you have very little data (a "batch" of only 3 episodes), which method—MC or TD—is more likely to converge to a "Certainty Equivalence" solution that leverages the Markov property to guess values for states it hasn't fully explored?
4.  **Bias vs. Variance:** Which of these two algorithms generally has higher variance, and why?

**Quick Answer Guide:**
*   *MC requires the episode to end to calculate the return G.*
*   *TD bootstraps by using the current estimate of the next state's value to update the current state.*
*   *TD(0) converges to the certainty equivalence solution in batch settings.*
*   *MC has higher variance because returns can fluctuate wildly between episodes.*